<a href="https://colab.research.google.com/github/ahmedosamasalem/Chest_X-Ray_Detector/blob/main/Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import tensorflow as tf
import keras
from google.colab import files

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 765.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 124.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 92.9 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0


/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'covid19-radiography-database' dataset.
Path to dataset files: /kaggle/input/covid19-radiography-database


In [ ]:
DATASET_DIR = os.path.join(
    path ,
    "COVID-19_Radiography_Dataset"
)

print("Dataset directory:", DATASET_DIR)

classes = ["COVID","Normal","Lung_Opacity","Viral Pneumonia"]

print("Classes:", classes)
print("Number of classes:", len(classes))


Dataset directory: /kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset
Classes: ['COVID', 'Normal', 'Lung_Opacity', 'Viral Pneumonia']
Number of classes: 4


In [ ]:
IMAGE_SIZE = (224,224)
BATCH_SIZE = 64

def load_images_from_folder(folder_path, label):
    images = []
    labels = []

    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    for filename in image_files:
        file_path = os.path.join(folder_path, filename)
        image = cv2.imread(file_path)                  # reads as BGR, 3 channels
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # convert to RGB (pretrained models expect RGB order)
        image = cv2.resize(image, IMAGE_SIZE)  # resize to the desired size
        images.append(image)
        labels.append(label)

    return images, labels

In [ ]:
COVID_DIR = os.path.join(DATASET_DIR, "COVID", "images")
NORMAL_DIR = os.path.join(DATASET_DIR, "Normal", "images")
LUNG_OPACITY_DIR = os.path.join(DATASET_DIR, "Lung_Opacity", "images")
VIRAL_PNEUMONIA_DIR = os.path.join(DATASET_DIR, "Viral Pneumonia", "images")

In [ ]:
print("COVID_DIR:", COVID_DIR)
print("NORMAL_DIR:", NORMAL_DIR)
print("LUNG_OPACITY_DIR:", LUNG_OPACITY_DIR)
print("VIRAL_PNEUMONIA_DIR:", VIRAL_PNEUMONIA_DIR)

COVID_DIR: /kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset/COVID/images
NORMAL_DIR: /kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset/Normal/images
LUNG_OPACITY_DIR: /kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset/Lung_Opacity/images
VIRAL_PNEUMONIA_DIR: /kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset/Viral Pneumonia/images


In [ ]:
print("Loading training covid images...")
covid_images, covid_labels = load_images_from_folder(COVID_DIR, 0)
print(f"Loaded {len(covid_images)} training covid images.")

print("Loading training normal images...")
normal_images, normal_labels = load_images_from_folder(NORMAL_DIR, 1)
print(f"Loaded {len(normal_images)} training normal images.")

print("Loading training lung opacity images...")
lung_opacity_images, lung_opacity_labels = load_images_from_folder(LUNG_OPACITY_DIR, 2)
print(f"Loaded {len(lung_opacity_images)} training lung opacity images.")

print("Loading training viral pneumonia images...")
viral_pneumonia_images, viral_pneumonia_labels = load_images_from_folder(VIRAL_PNEUMONIA_DIR, 3)
print(f"Loaded {len(viral_pneumonia_images)} training viral pneumonia images.")

Loading training covid images...
Loaded 3616 training covid images.
Loading training normal images...
Loaded 10192 training normal images.
Loading training lung opacity images...
Loaded 6012 training lung opacity images.
Loading training viral pneumonia images...
Loaded 1345 training viral pneumonia images.


In [ ]:
X = np.array(covid_images +normal_images +lung_opacity_images +viral_pneumonia_images)
y = np.array(covid_labels +normal_labels +lung_opacity_labels +viral_pneumonia_labels)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (21165, 224, 224, 3)
y shape: (21165,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.30,random_state=42,stratify=y)

print("Training:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Training: (14815, 224, 224, 3) (14815,)
Test: (6350, 224, 224, 3) (6350,)


In [ ]:
X_train = preprocess_input(X_train.astype('float32'))
X_test = preprocess_input(X_test.astype('float32'))

In [ ]:
base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3),include_top=False,weights="imagenet")
base_model.trainable = False   # Freeze the pretrained base so its weights don't get destroyed on the first pass
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(4, activation="softmax")
])


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,430,468 (9.27 MB)

 Trainable params: 172,484 (673.77 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
model.compile(optimizer="Adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])


In [ ]:
model.fit(X_train, y_train,batch_size=BATCH_SIZE,epochs=10,validation_split=0.2)


Epoch 1/10
186/186 ━━━━━━━━━━━━━━━━━━━━ 271s 1s/step - accuracy: 0.7554 - loss: 0.6387 - val_accuracy: 0.8458 - val_loss: 0.4120
Epoch 2/10
186/186 ━━━━━━━━━━━━━━━━━━━━ 280s 2s/step - accuracy: 0.8301 - loss: 0.4530 - val_accuracy: 0.8687 - val_loss: 0.3762
Epoch 3/10
186/186 ━━━━━━━━━━━━━━━━━━━━ 329s 2s/step - accuracy: 0.8480 - loss: 0.4068 - val_accuracy: 0.8731 - val_loss: 0.3510
Epoch 4/10
186/186 ━━━━━━━━━━━━━━━━━━━━ 320s 2s/step - accuracy: 0.8610 - loss: 0.3809 - val_accuracy: 0.8785 - val_loss: 0.3287
Epoch 5/10
186/186 ━━━━━━━━━━━━━━━━━━━━ 338s 2s/step - accuracy: 0.8626 - loss: 0.3638 - val_accuracy: 0.8923 - val_loss: 0.3086
Epoch 6/10
186/186 ━━━━━━━━━━━━━━━━━━━━ 347s 2s/step - accuracy: 0.8715 - loss: 0.3434 - val_accuracy: 0.8940 - val_loss: 0.2990
Epoch 7/10
186/186 ━━━━━━━━━━━━━━━━━━━━ 342s 2s/step - accuracy: 0.8804 - loss: 0.3265 - val_accuracy: 0.8869 - val_loss: 0.3026
Epoch 8/10
186/186 ━━━━━━━━━━━━━━━━━━━━ 354s 2s/step - accuracy: 0.8873 - loss: 0.3054 - val_accu

In [ ]:
model.save("model.keras")
print("Model saved successfully!")
files.download("model.keras")
print("Model downloaded successfully!")

Model saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model downloaded successfully!


In [ ]:
y_train_pred = model.predict(X_train, batch_size=32)
y_train_pred = np.argmax(y_train_pred, axis=1)
print(classification_report(y_train, y_train_pred))
del y_train_pred

y_test_pred = model.predict(X_test, batch_size=32)
y_test_pred = np.argmax(y_test_pred, axis=1)
print(classification_report(y_test, y_test_pred))


463/463 ━━━━━━━━━━━━━━━━━━━━ 258s 557ms/step
              precision    recall  f1-score   support

           0       0.98      0.85      0.91      2531
           1       0.86      0.97      0.91      7134
           2       0.95      0.80      0.87      4208
           3       0.96      0.99      0.97       942

    accuracy                           0.90     14815
   macro avg       0.94      0.90      0.92     14815
weighted avg       0.91      0.90      0.90     14815

199/199 ━━━━━━━━━━━━━━━━━━━━ 111s 558ms/step
              precision    recall  f1-score   support

           0       0.96      0.78      0.86      1085
           1       0.83      0.97      0.90      3058
           2       0.92      0.77      0.84      1804
           3       0.93      0.94      0.93       403

    accuracy                           0.88      6350
   macro avg       0.91      0.86      0.88      6350
weighted avg       0.89      0.88      0.88      6350

